# Extract the variants of a gene list from a large VCF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fbcouto/multimerge-genomics-research/blob/main/notebooks/files/extract_genes_from_vcf.ipynb)

**The problem.** You have a VCF of a whole genome or a cohort (millions of records) and a list of genes: a
diagnostic panel, the genes of a pathway, the hits of a screen. You want only the variants in those genes,
with a margin for promoters and splice sites, without loading the file into memory. The gene coordinates
must come from an annotation of the same genome build as the VCF.

**What you need.** A VCF (plain or `.gz`, any size); a list of official gene symbols; a GFF3 or GTF
annotation of the same build (for example the NCBI RefSeq GFF: MMGEN maps its accessions, such as
`NC_000001.11`, to chromosome names).

**What you get.** A BED file of the gene regions; a VCF (`.vcf.gz` + `.tbi`) with only the variants in them;
the number of variants per gene; and the same extraction done two ways (with and without an index).

In [ ]:
# Setup. Runs on Google Colab or on your own computer: on Colab, the first run installs MMGEN from PyPI
# (about a minute). Tables use pandas and plots matplotlib when installed (Colab has both); without
# them the notebook prints plain text and skips the plots.
import platform, subprocess, sys
from pathlib import Path

RELEASE = "https://github.com/fbcouto/multimerge-genomics-research/releases/download/v2.5.2"      # the demo packs
PACKAGE = "multimerge-genomics==2.5.2"   # from PyPI
try:
    import multimerge_genomics as mmg
except ImportError:
    if platform.system() != "Linux":
        print("MMGEN is not installed in this Python. Install it with:\n"
              "  python -m pip install multimerge-genomics\n"
              "and run this cell again.")
        raise
    print("Installing MMGEN (about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PACKAGE], check=True)
    import multimerge_genomics as mmg

IN_NOTEBOOK = "ipykernel" in sys.modules
try:
    from IPython.display import display, Markdown
except ImportError:                        # plain Python: print instead
    display, Markdown = print, str
try:
    import pandas as pd
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)
except ImportError:
    pd = None
try:
    import matplotlib
    if not IN_NOTEBOOK:
        matplotlib.use("Agg")              # plain Python: save the figures, open no window
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Chart colors (a palette checked for color-blind readers): two series, a two-sided scale, neutral ink
BLUE, ORANGE, RED, GRAY, INK = "#2a78d6", "#eb6834", "#e34948", "#a3a29c", "#52514e"


def show(rows, n=25):
    """Shows a list of dictionaries as a table (pandas when available, plain text otherwise)."""
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]:
            print("  ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > n:
        print(f"... and {len(rows) - n} more rows")


def tidy(ax):
    """Recessive axes: no top or right frame, a light grid behind the data."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)


def save_figure(fig, path):
    """Saves a figure; shows it in a notebook, closes it in plain Python."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Figure saved to", path)
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


print(mmg.engine_info())

## Data

By default, from the human demo pack (downloaded once, about 45 MB): the Genome in a Bottle (NIST) benchmark
calls of HG002, a public genome of a volunteer who consented to open sharing (cut to chromosome 22 and
the genes of some clinical panels, ±5 kb), and the NCBI RefSeq GRCh38 annotation of those genes.
To use your own files, set `MY_VCF` and `MY_ANNOTATION`, and write your genes in `GENES`.

In [ ]:
# Your files and genes (None = the demo data)
MY_VCF = None                # your VCF (plain or .gz, any size)
MY_ANNOTATION = None         # GFF3 or GTF of the SAME build as the VCF
GENES = ["LDLR", "APOB", "PCSK9", "LDLRAP1", "BRCA1", "BRCA2", "CFTR"]   # official symbols
PADDING = 2000               # bases added on each side of a gene (promoter, splice sites)

import urllib.request, zipfile

DATA = Path("data")                          # the demo files are extracted here
OUT = Path("results") / "extract_genes_from_vcf"         # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)


def demo_file(pack, member):
    """One file of a demo pack: downloads the pack once into the working folder and extracts only that file."""
    target = DATA / member
    if not target.exists():
        if not Path(pack).exists():
            print(f"Downloading {pack} from the MMGEN release...")
            urllib.request.urlretrieve(f"{RELEASE}/{pack}", pack + ".part")
            Path(pack + ".part").replace(pack)
        with zipfile.ZipFile(pack) as z:
            z.extract(member, DATA)
    return str(target)


USING_DEMO = MY_VCF is None and MY_ANNOTATION is None
VCF = MY_VCF or demo_file("mmgen_demo_human_v3.zip", "human/giab/HG002.vcf.gz")
ANNOTATION = MY_ANNOTATION or demo_file("mmgen_demo_human_v3.zip", "human/panel_genes.gff3.gz")
print("VCF:", VCF)
print("Annotation:", ANNOTATION)

## Step 1. From gene names to regions

`gene_regions` looks each symbol up in the annotation and returns its span (first to last base of the
gene), plus `PADDING` on each side, named as the VCF names its chromosomes. A symbol that is an old
alias or misspelled ends up in `not_found`; a gene also placed on alternative sequences (patches) is
reported, and the primary chromosome is used. The regions are also written as BED.

Watch the coordinates: BED starts at 0 and excludes the end's next base; region strings such as
`chr19:11084463-11138820` start at 1 and include the end, as in `samtools`, `tabix` and `bcftools`.

In [ ]:
naming = mmg.validate_file(VCF, max_lines=10_000)["naming"]
genes = mmg.gene_regions(ANNOTATION, GENES, output=str(OUT / "genes.bed"), padding=PADDING,
                         naming=naming if naming in ("chr", "no_chr") else None)
print(genes["text"])
if genes["not_found"]:
    print("NOT FOUND (check the spelling; use the official HGNC symbol):", genes["not_found"])
regions = {g["gene"]: f'{g["chrom"]}:{g["start"] + 1}-{g["end"]}' for g in genes["regions"]}   # BED -> 1-based
show({"gene": gene, "region (1-based)": region} for gene, region in regions.items())

## Step 2. An index for the VCF

An index (`.tbi`) lets MMGEN, `tabix` or `bcftools` read only the blocks of the file that hold a region,
instead of the whole file. It needs a file sorted by position and compressed as BGZF. If the VCF already
has its `.tbi`, it is used as it is; if it can be indexed, the index is written next to it; otherwise a
sorted BGZF copy is written, with its index (once, for a file of any size).

In [ ]:
INDEXED = VCF
if Path(VCF + ".tbi").exists():
    print("Index found:", VCF + ".tbi")
else:
    try:
        print(mmg.index_file(VCF))
    except Exception as err:          # not BGZF, or not sorted: write a sorted BGZF copy with its index
        print("Cannot index the file as it is:", err)
        INDEXED = str(OUT / "sorted.vcf.gz")
        records, runs, skipped = mmg.sort_genomic([VCF], INDEXED, index=True)
        print(f"Sorted copy: {INDEXED} ({records:,} records) + .tbi")

## Step 3. Extract through the index

`query_region` takes all the regions at once (overlapping ones are merged, so each record comes out once)
and writes them with the header. A record counts when it overlaps a region, so an indel that starts just
before a gene and reaches into it is kept.

In [ ]:
import time

t = time.perf_counter()
n_indexed = mmg.query_region(INDEXED, list(regions.values()), output=str(OUT / "genes.vcf.gz"))
print(f"{n_indexed:,} variants in the {len(regions)} gene regions ({time.perf_counter() - t:.2f} s) -> {OUT / 'genes.vcf.gz'}")
mmg.index_file(str(OUT / "genes.vcf.gz"))      # so IGV, tabix and bcftools can open the result directly

per_gene = [{"gene": gene, "region": region, "variants": len(mmg.query_region(INDEXED, region))}
            for gene, region in regions.items()]
show(per_gene)

## Step 4. The same without an index, and with a condition

`vcf_filter` streams the whole file: no index needed, and the regions can be combined with an expression
on the records. It is the better choice for a single pass with conditions; the index is better for many
questions on the same file. Both must give the same records.

In [ ]:
t = time.perf_counter()
streamed = mmg.vcf_filter(VCF, str(OUT / "genes_streamed.vcf.gz"), regions=list(regions.values()))
print(f"vcf_filter: {streamed['kept']:,} of {streamed['read']:,} records ({time.perf_counter() - t:.2f} s, whole file read)")

passing = mmg.vcf_filter(VCF, str(OUT / "genes_pass_snvs.vcf.gz"), regions=list(regions.values()),
                         expression='FILTER == "PASS" && TYPE == "snp"')
print(f"Only the PASS SNVs in those genes: {passing['kept']:,}")

print(mmg.vcf_stats(str(OUT / "genes.vcf.gz"))["text"])

## Check the result

In [ ]:
def check(condition, text):
    print(("OK     " if condition else "CHECK  ") + text)


check(not genes["not_found"], "every gene was found in the annotation")
check(n_indexed == streamed["kept"], f"index and stream agree ({n_indexed:,} and {streamed['kept']:,} records)")
check(sum(g["variants"] for g in per_gene) >= n_indexed, "per-gene counts add up (a variant in two overlapping genes counts twice)")

# every extracted record overlaps one of the regions
spans = [(g["chrom"], g["start"] + 1, g["end"]) for g in genes["regions"]]
outside = 0
for line in mmg.query_region(str(OUT / "genes.vcf.gz"), [r for r in regions.values()], header=False):
    chrom, pos, _, ref = line.split("\t", 4)[:4]
    start, end = int(pos), int(pos) + len(ref) - 1
    outside += not any(c == chrom and start <= e and end >= s for c, s, e in spans)
check(outside == 0, f"no record lies outside the regions ({outside})")
if USING_DEMO:
    assert n_indexed == streamed["kept"] > 500 and not genes["not_found"]
    print("Demo expectations met.")

## Methods paragraph

A starting text for the methods section of a paper or report, filled in with the version and the parameters
this run used. Edit it to your study, and cite the methods MMGEN reproduces (see References).

In [ ]:
import textwrap
VERSION = mmg.engine_info()["version"]
methods = f"""The coordinates of {len(regions)} genes ({', '.join(regions)}) were taken from the annotation {Path(ANNOTATION).name}
with gene_regions, extended by {PADDING:,} bases on each side. Variants overlapping these regions were extracted from the
BGZF-compressed, tabix-indexed VCF with query_region (and, as a check, by streaming the file with vcf_filter), using
MMGEN version {VERSION} (Couto F. MMGEN: MultiMerge Genomics (version {VERSION}). Zenodo. https://doi.org/10.5281/zenodo.20417492)."""
print(textwrap.fill(" ".join(methods.split()), 110))

## Adapt it to your data

* Use a panel's official symbols (HGNC for human); an alias such as `HER2` for `ERBB2` lands in `not_found`.
* Want only the exons? Write a BED of the exons from the GFF (feature `exon`) and pass those regions instead.
* A cohort VCF: add `samples=[...]` to `vcf_filter` to keep only some people.
* The GRCh38 RefSeq GFF is at NCBI (`GCF_000001405.40_GRCh38.p14_genomic.gff.gz`); for GRCh37 use that build's
  annotation. Check the VCF's build first (see `harmonize_two_vcfs.ipynb`).
* Keep `genes.vcf.gz` and its `.tbi`: they open directly in IGV.

## References

* O'Leary N.A. et al. (2016) Reference sequence (RefSeq) database at NCBI. *Nucleic Acids Research* 44:D733.
* Li H. (2011) Tabix: fast retrieval of sequence features from generic TAB-delimited files. *Bioinformatics* 27:718.
* Danecek P. et al. (2021) Twelve years of SAMtools and BCFtools. *GigaScience* 10:giab008 (region and expression conventions).
* Quinlan A.R., Hall I.M. (2010) BEDTools. *Bioinformatics* 26:841 (BED coordinates).
* Wagner J. et al. (2022) Benchmarking challenging small variants with linked and long reads. *Cell Genomics* 2:100128.